# Load USDA dataset with categories

## Setup
Import libraries needed throughout the notebook.

In [11]:
import json
import pandas as pd

## 1. USDA — Load & Pivot Macros
Load `food.csv` and `food_nutrient.csv` from the FoodData Central export. Filter to the four macronutrients (energy, protein, carbs, fat), pivot from long to wide so each food has one row, then merge with the food table.

In [ ]:
import pandas as pd

DATA_DIR = "raw_data/FoodData_Central_csv_2025-12-18"

# --- load tables --------------------------------------------------------
food = pd.read_csv(f"{DATA_DIR}/food.csv", usecols=["fdc_id", "description", "food_category_id"])
food_nutrient = pd.read_csv(f"{DATA_DIR}/food_nutrient.csv", usecols=["fdc_id", "nutrient_id", "amount"])

# --- nutrient IDs of interest -------------------------------------------
NUTRIENT_MAP = {
    1008: "energy_kcal",
    1003: "protein_g",
    1005: "carbohydrate_g",
    1004: "fat_g",
}

# keep only the four nutrients, then pivot to one column per nutrient
fn = food_nutrient[food_nutrient["nutrient_id"].isin(NUTRIENT_MAP)].copy()
fn["nutrient_name"] = fn["nutrient_id"].map(NUTRIENT_MAP)
fn_pivot = fn.pivot_table(index="fdc_id", columns="nutrient_name", values="amount", aggfunc="first")
fn_pivot.reset_index(inplace=True)

# --- merge with food table & rename category ----------------------------
df = food.merge(fn_pivot, on="fdc_id", how="inner")
df.rename(columns={"food_category_id": "category", "description": "food_description"}, inplace=True)

# --- drop rows where ANY of the five key columns is NaN -----------------
required = ["energy_kcal", "protein_g", "carbohydrate_g", "fat_g", "category"]
df.dropna(subset=required, inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()

(1826573, 7)


,fdc_id,food_description,category,carbohydrate_g,energy_kcal,fat_g,protein_g
0,1105904,WESSON Vegetable Oil 1 GAL,Oils Edible,0.00,867.0,93.33,0.00
1,1105905,SWANSON BROTH BEEF,Herbs/Spices/Extracts,0.42,4.0,0.00,0.83
2,1105906,CAMPBELL'S SLOW KETTLE SOUP CLAM CHOWDER,Prepared Soups,6.12,82.0,5.31,2.45
3,1105907,CAMPBELL'S SLOW KETTLE SOUP CHEESE BROCCOLI,Prepared Soups,5.31,82.0,6.12,1.22
4,1105908,SWANSON BROTH CHICKEN,Herbs/Spices/Extracts,0.42,4.0,0.00,0.83


In [15]:
df.drop(columns=["fdc_id"], inplace=True)

In [ ]:
df.category.nunique()

In [ ]:
# list all distinct categories
categories = df.category.unique()
print(categories)

In [ ]:
# surface categories that are just numeric IDs — those need resolving
categories_numeric = [cat for cat in categories if cat.isdigit()]
print(categories_numeric)
print(len(categories_numeric))

## 2. Resolve Numeric Category IDs
USDA food items reference category IDs as plain numbers. Map them to human-readable descriptions using `food_category.csv` (standard categories) and `wweia_food_category.csv` (survey categories).

In [18]:
# --- resolve numeric category IDs to descriptions ----------------------
food_cat = pd.read_csv(f"{DATA_DIR}/food_category.csv", dtype=str)
wweia_cat = pd.read_csv(f"{DATA_DIR}/wweia_food_category.csv", dtype=str)

# build a combined lookup: food_category id + wweia code -> description
cat_map = dict(zip(food_cat["id"], food_cat["description"]))
cat_map.update(dict(zip(wweia_cat["wweia_food_category"], wweia_cat["wweia_food_category_description"])))

In [19]:
# now replace the strings with only numbers in the "category" column in df with the resolved descriptions
df["category"] = df["category"].apply(lambda x: cat_map.get(x, x))

In [ ]:
df.category

### Export USDA data
Save the cleaned USDA dataframe to `usda_data_cats.csv`.

In [21]:
usda_data = df.copy()
usda_data.to_csv("usda_data_cats.csv", index=False)

# Load open food facts with categories

In [ ]:
import pandas as pd

cols = [
    "product_name",
    "main_category_en",
    "categories",
    "categories_en",
    "energy-kcal_100g",
    "fat_100g",
    "carbohydrates_100g",
    "proteins_100g",
]

off = pd.read_csv(
    "raw_data/en.openfoodfacts.org.products.csv",
    sep="\t",
    usecols=cols,
    low_memory=False,
)

off.dropna(subset=cols, inplace=True)
off.reset_index(drop=True, inplace=True)

print(off.shape)
off.head()

In [ ]:
off.to_csv("off_data_cats.csv", index=False)

In [ ]:
off.main_category_en.nunique()

## 3. Harmonise Columns
Rename both USDA and OFF columns to a shared schema (`item_name`, `cat`, `kcal_100g`, `protein_100g`, `carbs_100g`, `fat_100g`), add a `source` tag, and drop columns that are no longer needed.

In [ ]:
usda_data = pd.read_csv("usda_data_cats.csv")

In [ ]:
# make the columns match ['item_id', 'item_name', 'brand', 'kcal_100g', 'protein_100g',
#       'carbs_100g', 'fat_100g']
# and keep the added columns (which do not match the list above)
usda_data.rename(
    columns={
        "food_description": "item_name",
        "category": "cat",
        "energy_kcal": "kcal_100g",
        "protein_g": "protein_100g",
        "carbohydrate_g": "carbs_100g",
        "fat_g": "fat_100g",
    },
    inplace=True,
)

# also for off data
off.rename(
    columns={
        "product_name": "item_name",
        "main_category_en": "cat",
        "energy-kcal_100g": "kcal_100g",
        "proteins_100g": "protein_100g",
        "carbohydrates_100g": "carbs_100g",
        "fat_100g": "fat_100g",
    },
    inplace=True,
)


In [ ]:
# add a column to each dataframe called "source" with values "off" and "usda" respectively
off["source"] = "off"
usda_data["source"] = "usda"

In [ ]:
off_clean = off.copy()
usda_clean = usda_data.copy()
off_clean.drop(columns=["categories", "categories_en"], inplace=True)
usda_clean.drop(columns=["fdc_id"], inplace=True)

In [ ]:
print(off_clean.columns.tolist())
print(usda_clean.columns.tolist())

In [ ]:
off_clean.to_csv("off_data_clean2.csv", index=False)
usda_clean.to_csv("usda_data_clean2.csv", index=False)

## 4. Filter & Normalise Categories
Drop OFF rows whose `cat` column contains a non-English language tag (e.g. `fr:`, `de:`), then normalise all text to lowercase for consistent matching.

In [ ]:
# drop all the rows where "cat" contains a language tag (e.g. "fr:") since we cannot resolve those categories
off_clean = off_clean[~off_clean["cat"].str.contains(r"^[a-z]{2}:")].copy()
off_clean.reset_index(drop=True, inplace=True)

In [ ]:
# how many rows had a non-English language tag before dropping them?
off_lang_tag_count = off_clean.shape[0]

# breakdown of language prefixes found
prefixes = off[off["cat"].str.contains(r"^[a-z]{2}:", na=False)]["cat"].str.extract(r"^([a-z]{2}):", expand=False)
print(f"Rows with language tag: {len(prefixes)} / {len(off)}")
print(f"\nTop prefixes:\n{prefixes.value_counts().head(10).to_string()}")

In [ ]:
off_clean.info()

In [ ]:
# make all the entries lowercase in the 'cat' and 'item_name' columns for both dataframes
off_clean["cat"] = off_clean["cat"].str.lower()
off_clean["item_name"] = off_clean["item_name"].str.lower()
usda_clean["cat"] = usda_clean["cat"].str.lower()
usda_clean["item_name"] = usda_clean["item_name"].str.lower()

In [ ]:
# OFF — word count distribution in category labels
off_word_counts = off_clean.copy()
off_word_counts['word_count'] = off_word_counts['cat'].str.count(' ') + 1
print("OFF category word count distribution:")
print(off_word_counts['word_count'].value_counts().to_string())

In [ ]:
# USDA — word count distribution in category labels
usda_word_counts = usda_clean.copy()
usda_word_counts['word_count'] = usda_word_counts['cat'].str.count(' ') + 1
print("USDA category word count distribution:")
print(usda_word_counts['word_count'].value_counts().to_string())

In [ ]:
# how many distinct OFF categories have exactly 7 words?
off_word_counts[off_word_counts['word_count'] == 7].nunique()

In [ ]:
# which USDA categories have 6 words — and what items belong to them?
print(usda_clean[usda_word_counts['word_count'] == 6]['cat'].unique())
usda_word_counts.item_name[usda_word_counts['word_count'] == 6].head(20)

## 5. Category Exploration
Spot-check the granularity of category labels across both datasets. Key observations:
- Most higher word-count categories match only a few items
- Category labels carry more semantic signal than product names

In [ ]:
usda_clean.cat[usda_clean.cat.str.contains('cereal')].unique()

<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       'cereal',
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

There are f.e. 20 different cereal categories in the usda dataset. 

In [ ]:
off_clean.cat[off_clean.cat.str.contains('cereal')].unique()
# 84 in the off-dataset

<StringArray>
[                                                                                                                                                                       'yogurts with cereals',
                                                                                                                                                                           'breakfast cereals',
                                                                                                                                                                               'cereal flakes',
                                                                                                                                                                                 'cereal bars',
                                                                                                                                                                            'extruded cereals',
                          

## 6. Final Combined Dataset
Load the pre-built cleaned outputs (`usda_final.csv`, `off_nutrition_clean.csv`, `combined_final.csv`) for final inspection and spot-checks.

In [ ]:
import pandas as pd
off_clean = pd.read_csv("old/off_nutrition_clean.csv")
usda_clean =pd.read_csv("usda_final.csv")
combined = pd.read_csv("combined_final.csv")


/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_88470/3424309581.py:2: DtypeWarning: Columns (0: item_id) have mixed types. Specify dtype option on import or set low_memory=False.
  off_clean = pd.read_csv("old/off_nutrition_clean.csv")


In [ ]:
off_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 66341 entries, 0 to 66340
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       66341 non-null  object 
 1   item_name     66341 non-null  str    
 2   brand         56777 non-null  str    
 3   kcal_100g     66341 non-null  float64
 4   protein_100g  66341 non-null  float64
 5   carbs_100g    66341 non-null  float64
 6   fat_100g      66341 non-null  float64
 7   serving_size  47711 non-null  str    
 8   quantity      27935 non-null  str    
 9   source        66341 non-null  str    
dtypes: float64(4), object(1), str(5)
memory usage: 5.1+ MB


In [ ]:
off_clean.item_name[off_clean.item_name.str.contains('penne')]

3650                 Breaded white meat chicken with penne
18016    High protein penne Pasta With Italian Sausage ...
28356                               Protein + penne rigate
34790                                         penne rigate
35191                               Pâtes maïs & riz penne
35621           Poulet au pesto rosso penne et ratatouille
48300                                       kipfilet penne
54486                               Strapasta penne rigate
56312               Pâtes mini penne rigate piccolini 500g
Name: item_name, dtype: str

In [ ]:
usda_clean.item_name[usda_clean.item_name.str.contains('bolognese penne')]

2504    bolognese penne
Name: item_name, dtype: str

In [ ]:
combined[combined.item_name.str.contains('spaghetti bolognese')]

,item_name,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2
22398,spaghetti bolognese,139.0,3.7,18.0,6.8,off,condiments & sauces,seasoning & spices


In [ ]:
combined.columns

Index(['item_name', 'kcal_100g', 'fat_100g', 'carbs_100g', 'protein_100g',
       'source', 'cat_l1', 'cat_l2'],
      dtype='str')